# Trabalho Final ALC 2026.1 — Compressão de tokens via SVD

Notebook pedagógico que percorre o pipeline completo do trabalho:

**review IMDb → matriz de embeddings token-level `F` (T×D) → poda (6 operadores) → métricas algébricas + acurácia downstream → resultados do artigo.**

## Perguntas de pesquisa (metodologia)

1. Leverage score preserva acurácia competitiva frente a heurísticas locais?
2. `svd_energia` difere de `svd`?
3. $E_k^{\mathrm{energia}}$ permanece estável quando $\rho$ varia?

## Resultados-alvo (seção Resultados do artigo)

- Tabela de acurácia downstream por método e $\rho$
- $E_k^{\mathrm{energia}} \approx 0{,}951$ (SVD, invariante a $\rho$)
- Figura trade-off acurácia × compressão

**Dependências:** `numpy`, `matplotlib` (requirements.txt) + `datasets`, `transformers`, `torch` para IMDb.

In [ ]:
import csv
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Garantir imports quando o notebook abre fora de codigo/
_CODIGO = os.path.dirname(os.path.abspath("trabalho_final.ipynb")) if os.path.isfile("trabalho_final.ipynb") else None
if _CODIGO is None:
    for _candidato in (".", "Trabalho_Final/codigo", os.path.join(os.getcwd(), "Trabalho_Final/codigo")):
        if os.path.isfile(os.path.join(_candidato, "compressores.py")):
            _CODIGO = os.path.abspath(_candidato)
            break
if _CODIGO and os.getcwd() != _CODIGO:
    os.chdir(_CODIGO)
if _CODIGO and _CODIGO not in sys.path:
    sys.path.insert(0, _CODIGO)

import dados
import embeddings
import compressores as cp
import metricas
import experimento
import visualizacao
from compressores import COMPRESSORES
from experimento import (
    ORCAMENTOS,
    PASTA_RESULTADOS,
    agregar,
    rodar_grade,
    salvar_csv,
)

# True = reexecuta grade (~229 s CPU); False = usa cache se existir
FORCAR_REEXECUCAO = False
CSV_REFERENCIA = os.path.join(PASTA_RESULTADOS, "resultados.csv")

print("Diretório de trabalho:", os.getcwd())
print("Cache CSV:", CSV_REFERENCIA)

## 1. `dados.py` — Corpus IMDb

**Metodologia (§ Corpus):** subconjunto balanceado de **400 reviews** (200 negativas, 200 positivas), split estratificado 70/30 (semente 42). A unidade experimental é cada review → uma matriz $F^{(i)} \in \mathbb{R}^{T_i \times D}$.

**Por que este módulo?** Isola o carregamento do corpus e o split reprodutível, sem misturar lógica de embedding ou compressão.

In [ ]:
textos, rotulos = dados.carregar_imdb(n_por_classe=200, max_amostras=400)
idx_treino, idx_teste = dados.dividir_indices_estratificado(rotulos, semente=42)

n_neg = int(np.sum(rotulos == 0))
n_pos = int(np.sum(rotulos == 1))
print(f"Total: {len(textos)} reviews ({n_neg} neg, {n_pos} pos)")
print(f"Treino: {len(idx_treino)} | Teste: {len(idx_teste)}")
print(f"Treino por classe: neg={np.sum(rotulos[idx_treino]==0)}, pos={np.sum(rotulos[idx_treino]==1)}")
print(f"Teste por classe:  neg={np.sum(rotulos[idx_teste]==0)}, pos={np.sum(rotulos[idx_teste]==1)}")

## 2. `embeddings.py` — Construção da matriz $F$

**Metodologia (§ Construção da matriz):** encoder `all-MiniLM-L6-v2` via `last_hidden_state` (token-level, **não** pooling de sentença), remoção de [CLS]/[SEP]/padding, `max_tokens=256`, $D=384$. Cada linha = subtoken contextualizado; $\bar{T} \approx 209$ neste corpus.

**Por que este módulo?** Garante que podemos podar no eixo $T$ (linhas), preservando sequência token a token.

In [ ]:
F_exemplo = embeddings.texto_para_embedding(textos[0], max_tokens=256)
print("Review 0 — shape F (T x D):", F_exemplo.shape)
print("D (dimensão embedding):", F_exemplo.shape[1], "(esperado 384)")

# Amostra de T médio (10 reviews para não demorar; grade usa todas)
amostra_ts = [embeddings.texto_para_embedding(t, max_tokens=256).shape[0] for t in textos[:10]]
print(f"T médio (10 reviews): {np.mean(amostra_ts):.1f} — corpus completo: T̄ ≈ 209")

### 2.1 Demo sintética — ligação ALC (SVD retangular)

Antes da grade completa, usamos $F$ sintético de posto baixo para verificar conceitos de ALC_19/ALC_21 **sem** rodar o encoder 400 vezes.

In [ ]:
from embeddings import gerar_embedding_sintetico

F = gerar_embedding_sintetico(n_tokens=64, dim=48, posto=6, semente=0)
print("shape de F (T x D):", F.shape)
print("posto numérico de F:", np.linalg.matrix_rank(F))
print("F^T F simétrica:", np.allclose(F.T @ F, (F.T @ F).T))

In [ ]:
print("SVD + leverage scores em F sintético:")
for orcamento in [0.75, 0.5, 0.25, 0.125]:
    indices, F_hat, info = cp.comprimir_svd(F, orcamento)
    C = metricas.compressao(F.shape[0], len(indices))
    fid = metricas.fidelidade_reconstrucao(F, info["reconstrucao"])
    print(
        f"  rho={orcamento:.3f} | tokens {F.shape[0]}->{len(indices)} "
        f"| C={C:.3f} | fidelidade={fid:.3f} | k={info['k']}"
    )

In [ ]:
_, S, _ = np.linalg.svd(F, full_matrices=False)
eigs = np.linalg.eigvalsh(F.T @ F)
eigs = np.sort(eigs)[::-1]
print("sigma_j^2 = lambda_j(F^T F):", np.allclose(S**2, eigs[: len(S)], atol=1e-10))

## 3. `compressores.py` — Operadores de poda

**Metodologia (§ Baselines e SVD):** dado orçamento $\rho$, mantemos $T' = \max(1, \lfloor \rho T \rfloor)$ linhas. Seis operadores sob o **mesmo** $\rho$:

| Método | Critério |
|--------|----------|
| `full` | sem poda (teto) |
| `random` | $T'$ tokens aleatórios |
| `norm` | maior $\|\mathbf{f}_i\|_2$ |
| `first` | primeiros $T'$ tokens |
| `svd` | leverage $\ell_i/k$ |
| `svd_energia` | leverage ponderado por $\Sigma_k$ |

**Por que este módulo?** Centraliza os operadores algébricos; treino e teste passam pelo mesmo operador e $\rho$.

In [ ]:
F_demo = F_exemplo
rho_demo = 0.5
print(f"Demonstração em review real (T={F_demo.shape[0]}, rho={rho_demo}):\n")

for nome, compressor in COMPRESSORES.items():
    indices, F_podado, info = compressor(F_demo, rho_demo, semente=0)
    C = metricas.compressao(F_demo.shape[0], len(indices))
    extra = ""
    if nome in ("svd", "svd_energia"):
        extra = f" | k={info['k']} | E_k={metricas.energia_espectral_preservada(info):.3f}"
    print(f"  {nome:12s} -> T'={len(indices):3d} | C={C:.3f}{extra}")

## 4. `metricas.py` — Avaliação dual

**Metodologia (§ Protocolo):** separamos duas avaliações:

- **(A) Métricas algébricas:** compressão $C = 1 - T'/T$; para SVD, $E_k^{\mathrm{energia}}$ sobre $F_k$ e fidelidade $R_k$.
- **(B) Downstream:** pooling médio + normalização $L_2$ → classificador Ridge ($\lambda=1$) → acurácia.

**Ponto central:** $E_k^{\mathrm{energia}}$ mede o subespaço truncado $F_k$, **não** a qualidade da seleção de índices $S$.

In [ ]:
def pooling(F_mat):
    v = F_mat.mean(axis=0)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


# Mini-pipeline: 20 reviews, um método, um rho
n_mini = 20
textos_mini = textos[:n_mini]
rotulos_mini = rotulos[:n_mini]
idx_tr, idx_te = dados.dividir_indices_estratificado(rotulos_mini, semente=42)
amostras_mini = [embeddings.texto_para_embedding(t, max_tokens=256) for t in textos_mini]

rho_mini = 0.5
vetores_tr, vetores_te = [], []
for i, F_i in enumerate(amostras_mini):
    _, F_pod, info = cp.comprimir_svd(F_i, rho_mini)
    v = pooling(F_pod)
    if i in set(idx_tr):
        vetores_tr.append(v)
    else:
        vetores_te.append(v)

X_tr = np.vstack(vetores_tr)
X_te = np.vstack(vetores_te)
y_tr, y_te = rotulos_mini[idx_tr], rotulos_mini[idx_te]
previsto = metricas.classificador_ridge(X_tr, y_tr, X_te)
acc_mini = metricas.acuracia(y_te, previsto)

_, _, info_svd = cp.comprimir_svd(amostras_mini[0], rho_mini)
print(f"Mini-experimento (n={n_mini}, svd, rho={rho_mini}):")
print(f"  Acurácia downstream: {acc_mini:.3f}")
print(f"  Compressão C:          {metricas.compressao(amostras_mini[0].shape[0], int(round(rho_mini * amostras_mini[0].shape[0]))):.3f}")
print(f"  E_k^energia (review 0): {metricas.energia_espectral_preservada(info_svd):.3f}")
print(f"  R_k (review 0):         {metricas.fidelidade_reconstrucao(amostras_mini[0], info_svd['reconstrucao']):.3f}")

## 5. `experimento.py` — Grade completa

**Metodologia (§ Protocolo experimental):** grade **6 métodos × 4 $\rho$ × 3 sementes = 72 execuções**. Embeddings cacheados uma vez; métodos determinísticos consolidados na agregação; sementes afetam só `random`.

**Cache:** se `resultados/resultados.csv` existir e `FORCAR_REEXECUCAO=False`, carregamos o cache (~229 s economizados na primeira execução).

In [ ]:
def carregar_linhas_csv(caminho):
    """Converte CSV em dicts tipados para agregar()."""
    campos_float = {
        "orcamento", "acuracia_downstream", "energia_espectral",
        "fidelidade_reconstrucao", "compressao", "t_medio", "t_linha_medio",
        "custo_tempo_operador", "custo_flops_densa", "proxy_atencao", "acerto_exploratorio",
    }
    linhas = []
    with open(caminho, newline="") as f:
        for row in csv.DictReader(f):
            linha = dict(row)
            linha["semente"] = int(row["semente"])
            for campo in campos_float:
                val = row.get(campo, "")
                if val in ("", "nan", "NaN"):
                    linha[campo] = float("nan")
                else:
                    linha[campo] = float(val)
            linhas.append(linha)
    return linhas


if not FORCAR_REEXECUCAO and os.path.isfile(CSV_REFERENCIA):
    linhas = carregar_linhas_csv(CSV_REFERENCIA)
    print(f"Cache carregado: {CSV_REFERENCIA} ({len(linhas)} linhas)")
else:
    print("Executando grade completa (400 amostras, max_tokens=256)...")
    t0 = time.perf_counter()
    linhas = rodar_grade(max_amostras=400, max_tokens=256)
    os.makedirs(PASTA_RESULTADOS, exist_ok=True)
    salvar_csv(linhas, CSV_REFERENCIA)
    salvar_csv(linhas, os.path.join(PASTA_RESULTADOS, "resultados_imdb.csv"))
    print(f"Grade concluída em {time.perf_counter() - t0:.0f} s")
    print(f"CSV salvo: {CSV_REFERENCIA}")

agregado = agregar(linhas)
print(f"\nMétodos agregados: {sorted(agregado.keys())}")

## 6. Resultados — Tabelas, figura e validação

**Metodologia (§ Resultados):** comparamos acurácia downstream por método e $\rho$; reportamos $E_k^{\mathrm{energia}}$ para SVD; figura trade-off acurácia × compressão.

In [ ]:
def formatar_acuracia(ponto, metodo):
    acc = ponto["acuracia_media"]
    std = ponto["acuracia_desvio"]
    if metodo == "random" and std > 0:
        return f"{acc:.3f}±{std:.2f}"
    return f"{acc:.3f}"


def tabela_acuracia(agregado):
    metodos_ordem = ["full", "random", "norm", "first", "svd", "svd_energia"]
    cabecalho = f"{'Método':<12}" + "".join(f"rho={r:<6.3f}" for r in ORCAMENTOS)
    print("Tabela — Acurácia downstream (IMDb)")
    print(cabecalho)
    print("-" * len(cabecalho))
    for metodo in metodos_ordem:
        if metodo not in agregado:
            continue
        por_rho = {p["orcamento"]: p for p in agregado[metodo]}
        vals = [formatar_acuracia(por_rho[r], metodo) for r in ORCAMENTOS]
        print(f"{metodo:<12}" + "".join(f"{v:<12}" for v in vals))


tabela_acuracia(agregado)

print("\nReferência artigo (resultados.tex):")
print("full     0.717  0.717  0.717  0.717")
print("random   0.714±0.02  0.708  0.706±0.02  0.711±0.03")
print("norm     0.725  0.700  0.667  0.650")
print("first    0.742  0.758  0.725  0.683")
print("svd      0.733  0.717  0.683  0.675")
print("svd_energia  0.725  0.683  0.675  0.650")

In [ ]:
# E_k^energia e estatísticas do corpus
ek_vals = [
    p["energia_espectral_media"]
    for p in agregado["svd"]
    if not np.isnan(p["energia_espectral_media"])
]
ek_media = float(np.mean(ek_vals))
print(f"E_k^energia (svd, média sobre rho): {ek_media:.3f}  (artigo: 0.951)")

ponto_svd_0125 = next(p for p in agregado["svd"] if p["orcamento"] == 0.125)
print(f"T̄ (tokens/review):     {ponto_svd_0125['t_medio']:.1f}  (artigo: ~209)")
print(f"T̄' @ rho=0.125 (svd):  {ponto_svd_0125['t_linha_medio']:.1f}  (artigo: ~26)")

In [ ]:
# Figura trade-off (salva PNG + exibe inline)
caminho_fig = os.path.join(PASTA_RESULTADOS, "tradeoff_acerto_compressao.png")
visualizacao.salvar_grafico_tradeoff(agregado, caminho_fig)

plt.figure(figsize=(8, 5))
for metodo, pontos in sorted(agregado.items()):
    pts = sorted(pontos, key=lambda p: p["compressao"])
    x = [p["compressao"] for p in pts]
    y = [p["acuracia_media"] for p in pts]
    e = [p["acuracia_desvio"] for p in pts]
    plt.errorbar(
        x, y, yerr=e, marker="o", capsize=3,
        label=visualizacao.ROTULOS.get(metodo, metodo),
        color=visualizacao.CORES.get(metodo, "#333333"),
    )
plt.xlabel("Compressão (1 - T'/T)")
plt.ylabel("Acurácia downstream")
plt.title("Trade-off Acurácia × Compressão (baselines + SVD)")
plt.legend(fontsize=8, loc="best")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Figura salva: {caminho_fig}")

### Discussão (alinhada a resultados.tex)

- **`full`** (0,717) serve de teto; em $\rho=0{,}5$, **`svd`** (0,717) iguala o teto e **`first`** (0,758) supera levemente — reviews concentram sinal de sentimento no início.
- Em $\rho=0{,}125$, **`random`** (0,711±0,03) supera **`svd`** (0,675): a SVD **não domina** baselines de forma uniforme.
- $E_k^{\mathrm{energia}} \approx 0{,}951$ permanece **estável** frente a $\rho$: confirma que mede o subespaço $F_k$, não a seleção $S$.

In [ ]:
# Validação automática vs referência do artigo (tolerância para arredondamento LaTeX)
TOLERANCIA = 0.005

REFERENCIA = {
    "full":       {0.75: 0.717, 0.5: 0.717, 0.25: 0.717, 0.125: 0.717},
    "random":     {0.75: 0.714, 0.5: 0.708, 0.25: 0.706, 0.125: 0.711},
    "norm":       {0.75: 0.725, 0.5: 0.700, 0.25: 0.667, 0.125: 0.650},
    "first":      {0.75: 0.742, 0.5: 0.758, 0.25: 0.725, 0.125: 0.683},
    "svd":        {0.75: 0.733, 0.5: 0.717, 0.25: 0.683, 0.125: 0.675},
    "svd_energia": {0.75: 0.725, 0.5: 0.683, 0.25: 0.675, 0.125: 0.650},
}

falhas = []
for metodo, rhos in REFERENCIA.items():
    por_rho = {p["orcamento"]: p for p in agregado[metodo]}
    for rho, ref in rhos.items():
        obtido = por_rho[rho]["acuracia_media"]
        if not np.isclose(obtido, ref, atol=TOLERANCIA):
            falhas.append((metodo, rho, obtido, ref))

if not np.isclose(ek_media, 0.951, atol=TOLERANCIA):
    falhas.append(("E_k^energia", "—", ek_media, 0.951))

if falhas:
    print("DIVERGÊNCIAS detectadas (cache desatualizado?):")
    for metodo, rho, obt, ref in falhas:
        print(f"  {metodo} rho={rho}: obtido={obt:.3f}, ref={ref:.3f}")
    print("\nDefina FORCAR_REEXECUCAO = True e reexecute a célula da grade.")
else:
    print("Validação OK: resultados consistentes com resultados.tex (tol={}).".format(TOLERANCIA))